# 05 – Interaktív térkép és eredménydashboard

## 5.1. Célkitűzés

A notebook célja a végleges Ridge-modell területi és időbeli
eredményeinek interaktív bemutatása.

Az elemzés három fő vizualizációt tartalmaz:

1. területi előrejelzési hibák interaktív térképe;
2. valós és előrejelzett bérek területenként választható idősora;
3. összefoglaló interaktív eredménydashboard.

Az interaktív vizualizációk nem helyettesítik a szakdolgozat
statikus ábráit, hanem az eredmények részletesebb feltárását és
közérthető bemutatását támogatják.

In [1]:
from pathlib import Path
import json

import pandas as pd
import requests
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots


# ÚTVONALAK

BASE_DIR = Path.cwd()

PROCESSED = BASE_DIR / "processed"
ML_RESULTS = BASE_DIR / "results" / "machine_learning"

INTERACTIVE_RESULTS = (
    BASE_DIR
    / "results"
    / "interactive_visualization"
)

INTERACTIVE_FIGURES = (
    BASE_DIR
    / "figures"
    / "interactive"
)

GEODATA = BASE_DIR / "geodata"


for directory in [
    INTERACTIVE_RESULTS,
    INTERACTIVE_FIGURES,
    GEODATA
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


print("Az interaktív vizualizációs környezet létrejött.")

Az interaktív vizualizációs környezet létrejött.


In [2]:
FILES = {
    "regional_errors":
        ML_RESULTS
        / "KSH_final_ridge_regional_errors.csv",

    "ridge_error_data":
        ML_RESULTS
        / "KSH_final_ridge_error_data.csv",

    "cluster_profiles":
        PROCESSED
        / "KSH_teruleti_profilok_klaszterekkel.csv"
}


missing_files = [
    str(file_path)
    for file_path in FILES.values()
    if not file_path.exists()
]


if missing_files:
    raise FileNotFoundError(
        "A következő fájlok hiányoznak:\n- "
        + "\n- ".join(missing_files)
    )


def read_csv_clean(file_path):
    """CSV betöltése és a névtelen indexoszlopok eltávolítása."""

    data = pd.read_csv(
        file_path,
        encoding="utf-8-sig"
    )

    unnamed_columns = [
        column
        for column in data.columns
        if str(column).startswith("Unnamed:")
    ]

    return data.drop(
        columns=unnamed_columns,
        errors="ignore"
    )


regional_errors = read_csv_clean(
    FILES["regional_errors"]
)

ridge_error_data = read_csv_clean(
    FILES["ridge_error_data"]
)

cluster_profiles = read_csv_clean(
    FILES["cluster_profiles"]
)


ridge_error_data["Cél_Dátum"] = pd.to_datetime(
    ridge_error_data["Cél_Dátum"]
)


assert len(regional_errors) == 20
assert len(ridge_error_data) == 160

assert ridge_error_data[
    "Területi egység neve"
].nunique() == 20

assert ridge_error_data[
    "Cél_Dátum"
].nunique() == 8


print("Az interaktív ábrákhoz szükséges adatok betöltődtek.")

Az interaktív ábrákhoz szükséges adatok betöltődtek.


In [3]:
cluster_lookup = (
    cluster_profiles[
        [
            "Területi egység neve",
            "Klaszter",
            "Klaszter_neve"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


assert len(cluster_lookup) == 20

assert not cluster_lookup[
    "Területi egység neve"
].duplicated().any()


regional_map_data = (
    regional_errors
    .merge(
        cluster_lookup,
        on="Területi egység neve",
        how="left",
        validate="one_to_one"
    )
)


assert len(regional_map_data) == 20

assert regional_map_data[
    [
        "Klaszter",
        "Klaszter_neve"
    ]
].notna().all().all()


display(
    regional_map_data[
        [
            "Területi egység neve",
            "Klaszter_neve",
            "Átlagos_valós_bér_Ft",
            "MAE_Ft",
            "MAPE_%",
            "Bias_Ft"
        ]
    ]
    .sort_values("MAE_Ft")
    .round(2)
)


print("A térképes eredménytábla elkészült.")

,Területi egység neve,Klaszter_neve,Átlagos_valós_bér_Ft,MAE_Ft,MAPE_%,Bias_Ft
0,Baranya,Alacsonyabb bérszint / magasabb munkanélküliség,579368.0,5693.40,0.98,4451.65
1,Békés,Alacsonyabb bérszint / magasabb munkanélküliség,513258.5,6557.07,1.30,6557.07
2,Jász-Nagykun-Szolnok,Alacsonyabb bérszint / magasabb munkanélküliség,554305.5,7155.54,1.29,6373.62
3,Szabolcs-Szatmár-Bereg,Alacsonyabb bérszint / magasabb munkanélküliség,499500.0,7250.80,1.44,4956.43
4,Somogy,Alacsonyabb bérszint / magasabb munkanélküliség,550373.0,8311.60,1.50,7972.21
5,Csongrád-Csanád,Magasabb bérszint / alacsonyabb munkanélküliség,586724.0,8453.01,1.44,6208.92
6,Bács-Kiskun,Magasabb bérszint / alacsonyabb munkanélküliség,581498.0,8702.32,1.49,7227.64
7,Pest,Magasabb bérszint / alacsonyabb munkanélküliség,618499.5,8959.67,1.46,7197.65
8,Hajdú-Bihar,Alacsonyabb bérszint / magasabb munkanélküliség,592346.5,9030.58,1.54,5118.62
9,Borsod-Abaúj-Zemplén,Alacsonyabb bérszint / magasabb munkanélküliség,547353.5,9118.83,1.69,8382.09


A térképes eredménytábla elkészült.


In [4]:
NUTS3_BY_REGION = {
    "Budapest": "HU110",
    "Pest": "HU120",
    "Fejér": "HU211",
    "Komárom-Esztergom": "HU212",
    "Veszprém": "HU213",
    "Győr-Moson-Sopron": "HU221",
    "Vas": "HU222",
    "Zala": "HU223",
    "Baranya": "HU231",
    "Somogy": "HU232",
    "Tolna": "HU233",
    "Borsod-Abaúj-Zemplén": "HU311",
    "Heves": "HU312",
    "Nógrád": "HU313",
    "Hajdú-Bihar": "HU321",
    "Jász-Nagykun-Szolnok": "HU322",
    "Szabolcs-Szatmár-Bereg": "HU323",
    "Bács-Kiskun": "HU331",
    "Békés": "HU332",
    "Csongrád-Csanád": "HU333"
}


regional_map_data["NUTS_ID"] = (
    regional_map_data[
        "Területi egység neve"
    ]
    .map(NUTS3_BY_REGION)
)


missing_nuts_regions = (
    regional_map_data.loc[
        regional_map_data["NUTS_ID"].isna(),
        "Területi egység neve"
    ]
    .tolist()
)


assert not missing_nuts_regions, (
    "Hiányzó NUTS-kódok: "
    + ", ".join(missing_nuts_regions)
)

assert regional_map_data["NUTS_ID"].nunique() == 20


display(
    regional_map_data[
        [
            "Területi egység neve",
            "NUTS_ID"
        ]
    ]
    .sort_values("NUTS_ID")
)


print("Mind a 20 terület rendelkezik NUTS 3 azonosítóval.")

,Területi egység neve,NUTS_ID
15,Budapest,HU110
7,Pest,HU120
14,Fejér,HU211
17,Komárom-Esztergom,HU212
10,Veszprém,HU213
18,Győr-Moson-Sopron,HU221
16,Vas,HU222
12,Zala,HU223
0,Baranya,HU231
4,Somogy,HU232


Mind a 20 terület rendelkezik NUTS 3 azonosítóval.


In [5]:
GEOJSON_URL = (
    "https://gisco-services.ec.europa.eu/"
    "distribution/v2/nuts/geojson/"
    "NUTS_RG_03M_2024_4326_LEVL_3.geojson"
)


GEOJSON_FILE = (
    GEODATA
    / "NUTS_RG_03M_2024_4326_LEVL_3.geojson"
)


if not GEOJSON_FILE.exists():

    response = requests.get(
        GEOJSON_URL,
        timeout=60
    )

    response.raise_for_status()

    GEOJSON_FILE.write_bytes(
        response.content
    )

    print("A NUTS 3 GeoJSON-fájl letöltődött.")

else:
    print("A NUTS 3 GeoJSON-fájl már rendelkezésre áll.")


with open(
    GEOJSON_FILE,
    "r",
    encoding="utf-8"
) as geojson_file:

    europe_nuts3 = json.load(
        geojson_file
    )


hungary_features = [
    feature
    for feature in europe_nuts3["features"]
    if feature["properties"].get("CNTR_CODE") == "HU"
]


hungary_nuts3 = {
    "type": "FeatureCollection",
    "features": hungary_features
}


geojson_nuts_ids = {
    feature["properties"]["NUTS_ID"]
    for feature in hungary_features
}


data_nuts_ids = set(
    regional_map_data["NUTS_ID"]
)


assert len(hungary_features) == 20

assert data_nuts_ids == geojson_nuts_ids, {
    "Hiányzik_a_térképről":
        sorted(data_nuts_ids - geojson_nuts_ids),

    "Hiányzik_az_adatokból":
        sorted(geojson_nuts_ids - data_nuts_ids)
}


print(
    "A 20 magyar terület és a térképi geometria "
    "illesztése sikeres."
)

A NUTS 3 GeoJSON-fájl már rendelkezésre áll.
A 20 magyar terület és a térképi geometria illesztése sikeres.


In [6]:
fig_map = px.choropleth(
    regional_map_data,
    geojson=hungary_nuts3,
    locations="NUTS_ID",
    featureidkey="properties.NUTS_ID",
    color="MAE_Ft",
    hover_name="Területi egység neve",

    hover_data={
        "NUTS_ID": False,
        "Klaszter_neve": True,
        "Átlagos_valós_bér_Ft": ":,.0f",
        "MAE_Ft": ":,.0f",
        "MAPE_%": ":.2f",
        "Bias_Ft": ":,.0f"
    },

    labels={
        "Klaszter_neve": "Gazdasági klaszter",
        "Átlagos_valós_bér_Ft": "Átlagos valós bér (Ft)",
        "MAE_Ft": "MAE (Ft)",
        "MAPE_%": "MAPE (%)",
        "Bias_Ft": "Bias (Ft)"
    },

    color_continuous_scale="YlOrRd",

    title=(
        "A Ridge-modell területi előrejelzési hibája, "
        "2024–2025"
    )
)


fig_map.update_geos(
    fitbounds="locations",
    visible=False
)


fig_map.update_traces(
    marker_line_color="white",
    marker_line_width=0.8
)


fig_map.update_layout(
    template="plotly_white",

    title={
        "x": 0.5,
        "xanchor": "center"
    },

    coloraxis_colorbar={
        "title": "MAE<br>(Ft)",
        "thickness": 18
    },

    margin={
        "l": 10,
        "r": 10,
        "t": 70,
        "b": 10
    },

    height=720
)


fig_map.show()


MAP_HTML = (
    INTERACTIVE_FIGURES
    / "01_interaktiv_regionalis_hibaterkep.html"
)


fig_map.write_html(
    MAP_HTML,
    include_plotlyjs=True,
    full_html=True
)


print(
    "Az interaktív térkép elmentve:",
    MAP_HTML
)

Az interaktív térkép elmentve: c:\Users\wldbe\Downloads\szakdolgozat12-20260820T154901Z-1-001\szakdolgozat12\figures\interactive\01_interaktiv_regionalis_hibaterkep.html


### A területi hibatérkép értelmezése

Az interaktív térkép alapján a Ridge-modell előrejelzési
pontossága területileg nem egyenletes. A legnagyobb átlagos
abszolút hibával Tolna és Győr-Moson-Sopron, míg a legkisebb
hibával Baranya és Békés rendelkezik.

A térkép elsődlegesen az abszolút hibát, vagyis a MAE értékét
ábrázolja. Mivel a területek bérszintje jelentősen különbözik,
a területi teljesítmény értékelésekor a relatív MAPE-mutatót
is figyelembe kell venni. Az egérmutatóval megjeleníthető
adatok ezért a bérszintet, a MAE-, MAPE- és biasértékeket,
valamint a klaszterbesorolást is tartalmazzák.

A vizualizáció alapján a magasabb bérszint önmagában nem
magyarázza teljesen az előrejelzési nehézséget. Az eltérésekben
területspecifikus gazdasági sajátosságok és rövid távú
idősoros változások is szerepet játszhatnak.

**Térképi adatforrás:** Eurostat GISCO, NUTS 2024,
NUTS 3 régiók, 1:3 milliós méretarány, EPSG:4326.

## 5.2. Kiválasztott terület korrelációs térképe

Az interaktív térkép a kiválasztott területi egység és a többi
terület éves bérváltozásának Pearson-korrelációját mutatja.
A legördülő listában Budapest vagy bármely vármegye kiválasztható.

A pozitív érték azonos irányú, a negatív érték ellentétes irányú
együttmozgást jelent. A korreláció leíró kapcsolat, ezért nem
értelmezhető automatikusan oksági vagy előrejelző hatásként.

In [19]:
# KIVÁLASZTOTT TERÜLET INTERAKTÍV KORRELÁCIÓS TÉRKÉPE

CORRELATION_FILE = (
    PROCESSED
    / "KSH_teruleti_YoY_bernovekedes_korrelacio_long.csv"
)

if not CORRELATION_FILE.exists():
    raise FileNotFoundError(
        "Nem található a korrelációs adatbázis. "
        "Először futtasd le a 02-es notebook új "
        "területi korrelációs részét: "
        f"{CORRELATION_FILE}"
    )


correlation_map_data = read_csv_clean(
    CORRELATION_FILE
)


required_correlation_columns = {
    "Kiválasztott_terület",
    "Összehasonlított_terület",
    "Pearson_r",
    "Spearman_r",
    "Megfigyelések_száma",
    "Kapcsolat_erőssége"
}

missing_correlation_columns = (
    required_correlation_columns
    -
    set(correlation_map_data.columns)
)

assert not missing_correlation_columns, (
    "Hiányzó korrelációs oszlopok: "
    + ", ".join(sorted(missing_correlation_columns))
)


correlation_map_data["NUTS_ID"] = (
    correlation_map_data[
        "Összehasonlított_terület"
    ]
    .map(NUTS3_BY_REGION)
)


assert not correlation_map_data["NUTS_ID"].isna().any()
assert len(correlation_map_data) == 400

assert (
    correlation_map_data
    .groupby("Kiválasztott_terület")
    .size()
    .eq(20)
    .all()
)


DEFAULT_CORRELATION_REGION = "Budapest"

correlation_regions = sorted(
    correlation_map_data[
        "Kiválasztott_terület"
    ]
    .unique()
)

correlation_region_order = (
    [DEFAULT_CORRELATION_REGION]
    +
    [
        region
        for region in correlation_regions
        if region != DEFAULT_CORRELATION_REGION
    ]
)


fig_correlation_map = go.Figure()


for selected_region in correlation_region_order:

    selected_map_data = (
        correlation_map_data.loc[
            correlation_map_data[
                "Kiválasztott_terület"
            ] == selected_region
        ]
        .sort_values("NUTS_ID")
        .copy()
    )

    assert len(selected_map_data) == 20

    fig_correlation_map.add_trace(
        go.Choropleth(
            geojson=hungary_nuts3,
            locations=selected_map_data["NUTS_ID"],
            featureidkey="properties.NUTS_ID",
            z=selected_map_data["Pearson_r"],
            text=selected_map_data[
                "Összehasonlított_terület"
            ],
            customdata=selected_map_data[
                [
                    "Spearman_r",
                    "Megfigyelések_száma",
                    "Kapcsolat_erőssége"
                ]
            ].to_numpy(),
            coloraxis="coloraxis",
            marker_line_color="white",
            marker_line_width=0.8,
            visible=(
                selected_region
                ==
                DEFAULT_CORRELATION_REGION
            ),
            name=selected_region,
            hovertemplate=(
                "<b>%{text}</b><br>"
                + f"Kiválasztott terület: {selected_region}<br>"
                + "Pearson-korreláció: %{z:.3f}<br>"
                + "Spearman-korreláció: %{customdata[0]:.3f}<br>"
                + "Megfigyelések: %{customdata[1]}<br>"
                + "Kapcsolat erőssége: %{customdata[2]}"
                + "<extra></extra>"
            )
        )
    )

In [20]:
# LEGÖRDÜLŐ TERÜLETVÁLASZTÓ

correlation_map_buttons = []


for selected_region in correlation_region_order:

    visibility_mask = [
        region == selected_region
        for region in correlation_region_order
    ]

    correlation_map_buttons.append(
        {
            "label": selected_region,
            "method": "update",
            "args": [
                {
                    "visible": visibility_mask
                },
                {
                    "title": {
                        "text": (
                            "Éves bérváltozás korrelációja – "
                            + selected_region
                            + " és a többi terület"
                        ),
                        "x": 0.5,
                        "xanchor": "center"
                    }
                }
            ]
        }
    )


fig_correlation_map.update_geos(
    fitbounds="locations",
    visible=False
)


fig_correlation_map.update_layout(
    template="plotly_white",

    title={
        "text": (
            "Éves bérváltozás korrelációja – "
            "Budapest és a többi terület"
        ),
        "x": 0.5,
        "xanchor": "center"
    },

    coloraxis={
        "colorscale": "RdBu",
        "cmin": -1,
        "cmax": 1,
        "cmid": 0,
        "colorbar": {
            "title": "Pearson r",
            "thickness": 18
        }
    },

    updatemenus=[
        {
            "buttons": correlation_map_buttons,
            "active": 0,
            "direction": "down",
            "x": 0,
            "y": 1.12,
            "xanchor": "left",
            "yanchor": "top",
            "showactive": True
        }
    ],

    annotations=[
        {
            "text": "<b>Kiválasztott terület:</b>",
            "x": 0,
            "y": 1.18,
            "xref": "paper",
            "yref": "paper",
            "showarrow": False,
            "xanchor": "left"
        }
    ],

    margin={
        "l": 10,
        "r": 10,
        "t": 120,
        "b": 10
    },

    height=760
)


assert len(fig_correlation_map.data) == 20
assert len(correlation_map_buttons) == 20


fig_correlation_map.show()

In [21]:
CORRELATION_MAP_HTML = (
    INTERACTIVE_FIGURES
    / "04_interaktiv_teruleti_korrelacios_terkep.html"
)


fig_correlation_map.write_html(
    CORRELATION_MAP_HTML,
    include_plotlyjs=True,
    full_html=True
)


assert CORRELATION_MAP_HTML.exists()


print(
    "Az interaktív korrelációs térkép elmentve:",
    CORRELATION_MAP_HTML
)

Az interaktív korrelációs térkép elmentve: c:\Users\wldbe\Downloads\szakdolgozat12-20260820T154901Z-1-001\szakdolgozat12\figures\interactive\04_interaktiv_teruleti_korrelacios_terkep.html


## 5.2. Valós és előrejelzett bérek interaktív összehasonlítása

A következő vizualizáció lehetővé teszi a vizsgált területi
egység kiválasztását. Az ábra területenként mutatja be a
2024–2025-ös végső tesztidőszak valós és a Ridge-modell által
előrejelzett bruttó átlagbéreit.

A területválasztás segítségével megvizsgálható, hogy a modell
mely negyedévekben és milyen irányban tért el a tényleges
értékektől.

In [7]:
required_time_columns = {
    "Cél_Dátum",
    "Területi egység neve",
    "Valós_bér",
    "Predikció",
    "Hiba",
    "Abszolút_hiba",
    "APE_%"
}


assert required_time_columns.issubset(
    ridge_error_data.columns
)


ridge_time_data = (
    ridge_error_data[
        [
            "Cél_Dátum",
            "Területi egység neve",
            "Valós_bér",
            "Predikció",
            "Hiba",
            "Abszolút_hiba",
            "APE_%"
        ]
    ]
    .copy()
    .sort_values(
        [
            "Területi egység neve",
            "Cél_Dátum"
        ]
    )
    .reset_index(drop=True)
)


ridge_time_data["Negyedév"] = (
    ridge_time_data["Cél_Dátum"]
    .dt.year
    .astype(str)
    +
    " Q"
    +
    ridge_time_data["Cél_Dátum"]
    .dt.quarter
    .astype(str)
)


# ELLENŐRZÉSEK

assert len(ridge_time_data) == 160

assert ridge_time_data[
    "Területi egység neve"
].nunique() == 20

assert ridge_time_data[
    "Cél_Dátum"
].nunique() == 8

assert (
    ridge_time_data
    .groupby("Területi egység neve")
    .size()
    .eq(8)
    .all()
)

assert not ridge_time_data[
    [
        "Területi egység neve",
        "Cél_Dátum"
    ]
].duplicated().any()

assert ridge_time_data[
    [
        "Valós_bér",
        "Predikció",
        "Hiba",
        "Abszolút_hiba",
        "APE_%"
    ]
].notna().all().all()


time_data_preview = (
    ridge_time_data
    .head(8)
    .copy()
)


numeric_preview_columns = [
    "Valós_bér",
    "Predikció",
    "Hiba",
    "Abszolút_hiba",
    "APE_%"
]


time_data_preview[
    numeric_preview_columns
] = time_data_preview[
    numeric_preview_columns
].round(2)


display(time_data_preview)


print(
    "A területenkénti interaktív idősor "
    "adatai elkészültek."
)

,Cél_Dátum,Területi egység neve,Valós_bér,Predikció,Hiba,Abszolút_hiba,APE_%,Negyedév
0,2024-01-01,Baranya,528655.0,523809.86,-4845.14,4845.14,0.92,2024 Q1
1,2024-04-01,Baranya,545337.0,545215.14,-121.86,121.86,0.02,2024 Q2
2,2024-07-01,Baranya,546755.0,550995.12,4240.12,4240.12,0.78,2024 Q3
3,2024-10-01,Baranya,584097.0,590444.58,6347.58,6347.58,1.09,2024 Q4
4,2025-01-01,Baranya,581778.0,600883.49,19105.49,19105.49,3.28,2025 Q1
5,2025-04-01,Baranya,604250.0,609423.62,5173.62,5173.62,0.86,2025 Q2
6,2025-07-01,Baranya,603883.0,606660.35,2777.35,2777.35,0.46,2025 Q3
7,2025-10-01,Baranya,640189.0,643125.07,2936.07,2936.07,0.46,2025 Q4


A területenkénti interaktív idősor adatai elkészültek.


In [8]:
DEFAULT_REGION = "Budapest"


all_regions = sorted(
    ridge_time_data[
        "Területi egység neve"
    ].unique()
)


region_order = (
    [DEFAULT_REGION]
    +
    [
        region
        for region in all_regions
        if region != DEFAULT_REGION
    ]
)


target_dates = (
    ridge_time_data[
        [
            "Cél_Dátum",
            "Negyedév"
        ]
    ]
    .drop_duplicates()
    .sort_values("Cél_Dátum")
)


fig_region = go.Figure()

trace_indices = {}


for region_name in region_order:

    region_data = (
        ridge_time_data.loc[
            ridge_time_data[
                "Területi egység neve"
            ]
            ==
            region_name
        ]
        .sort_values("Cél_Dátum")
    )

    is_default_region = (
        region_name == DEFAULT_REGION
    )

    actual_trace_index = len(
        fig_region.data
    )


    # VALÓS BÉR

    fig_region.add_trace(
        go.Scatter(
            x=region_data["Cél_Dátum"],
            y=region_data["Valós_bér"],

            customdata=region_data[
                ["Negyedév"]
            ].to_numpy(),

            mode="lines+markers",
            name="Valós bér",

            line={
                "color": "#111111",
                "width": 3
            },

            marker={
                "size": 8,
                "symbol": "circle"
            },

            visible=is_default_region,

            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Valós bér: %{y:,.0f} Ft"
                "<extra>Valós bér</extra>"
            )
        )
    )


    prediction_trace_index = len(
        fig_region.data
    )


    # RIDGE-PREDIKCIÓ

    fig_region.add_trace(
        go.Scatter(
            x=region_data["Cél_Dátum"],
            y=region_data["Predikció"],

            customdata=region_data[
                [
                    "Negyedév",
                    "Hiba",
                    "Abszolút_hiba",
                    "APE_%"
                ]
            ].to_numpy(),

            mode="lines+markers",
            name="Ridge-predikció",

            line={
                "color": "#0072B2",
                "width": 3,
                "dash": "dash"
            },

            marker={
                "size": 8,
                "symbol": "square"
            },

            visible=is_default_region,

            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Ridge-predikció: %{y:,.0f} Ft<br>"
                "Irányított hiba: "
                "%{customdata[1]:+,.0f} Ft<br>"
                "Abszolút hiba: "
                "%{customdata[2]:,.0f} Ft<br>"
                "APE: %{customdata[3]:.2f}%"
                "<extra>Ridge-predikció</extra>"
            )
        )
    )


    trace_indices[region_name] = (
        actual_trace_index,
        prediction_trace_index
    )

In [9]:
region_buttons = []


for region_name in region_order:

    visibility_mask = [
        False
    ] * len(fig_region.data)


    actual_index, prediction_index = (
        trace_indices[region_name]
    )


    visibility_mask[
        actual_index
    ] = True

    visibility_mask[
        prediction_index
    ] = True


    region_buttons.append(
        {
            "label": region_name,
            "method": "update",

            "args": [
                {
                    "visible":
                        visibility_mask
                },

                {
                    "title.text": (
                        "Valós és előrejelzett bruttó átlagbér – "
                        + region_name
                    ),

                    "yaxis.autorange": True
                }
            ]
        }
    )


fig_region.update_layout(
    template="plotly_white",

    title={
        "text": (
            "Valós és előrejelzett bruttó átlagbér – "
            + DEFAULT_REGION
        ),
        "x": 0.5,
        "xanchor": "center"
    },

    updatemenus=[
        {
            "buttons": region_buttons,
            "active": 0,
            "direction": "down",
            "showactive": True,
            "x": 0.13,
            "y": 1.17,
            "xanchor": "left",
            "yanchor": "top"
        }
    ],

    annotations=[
        {
            "text": "<b>Területi egység:</b>",
            "x": 0,
            "y": 1.145,
            "xref": "paper",
            "yref": "paper",
            "showarrow": False,
            "xanchor": "left"
        }
    ],

    xaxis={
        "title": "Előrejelzett célnegyedév",
        "tickmode": "array",

        "tickvals":
            target_dates["Cél_Dátum"],

        "ticktext":
            target_dates["Negyedév"],

        "tickangle": -35,
        "showgrid": True
    },

    yaxis={
        "title": "Átlagos bruttó bér (Ft)",
        "tickformat": ",.0f",
        "autorange": True,
        "showgrid": True
    },

    legend={
        "orientation": "h",
        "x": 1,
        "y": 1.11,
        "xanchor": "right",
        "yanchor": "top"
    },

    hovermode="x unified",

    margin={
        "l": 80,
        "r": 40,
        "t": 135,
        "b": 90
    },

    height=650
)


assert len(fig_region.data) == 40
assert len(region_buttons) == 20


fig_region.show()

In [10]:
REGION_CHART_HTML = (
    INTERACTIVE_FIGURES
    / "02_interaktiv_valos_vs_ridge_teruletenkent.html"
)


fig_region.write_html(
    REGION_CHART_HTML,
    include_plotlyjs=True,
    full_html=True
)


ridge_time_data.to_csv(
    INTERACTIVE_RESULTS
    / "KSH_interaktiv_teruleti_idosor_adatok.csv",
    index=False,
    encoding="utf-8-sig"
)


assert REGION_CHART_HTML.exists()


print(
    "A területválasztós interaktív idősor elmentve:",
    REGION_CHART_HTML
)

A területválasztós interaktív idősor elmentve: c:\Users\wldbe\Downloads\szakdolgozat12-20260820T154901Z-1-001\szakdolgozat12\figures\interactive\02_interaktiv_valos_vs_ridge_teruletenkent.html


## 5.3. Interaktív területi eredménydashboard

A dashboard egyetlen felületen foglalja össze a végleges
Ridge-modell területenkénti teljesítményét.

A kiválasztott területhez három összefoglaló hibamutató
jelenik meg:

- az átlagos abszolút hiba (MAE);
- az átlagos abszolút százalékos hiba (MAPE);
- az átlagos irányított hiba (bias).

A dashboard emellett bemutatja a valós és előrejelzett bérek
időbeli alakulását, valamint az egyes célnegyedévek abszolút
előrejelzési hibáját. A terület kiválasztása kizárólag a
megjelenített eredményeket módosítja; a modell újratanítása
nem történik meg.

In [11]:
dashboard_summary = (
    regional_map_data[
        [
            "Területi egység neve",
            "Klaszter",
            "Klaszter_neve",
            "Átlagos_valós_bér_Ft",
            "MAE_Ft",
            "MAPE_%",
            "Bias_Ft"
        ]
    ]
    .copy()
    .set_index("Területi egység neve")
)


assert len(dashboard_summary) == 20

assert not dashboard_summary.index.duplicated().any()

assert set(
    dashboard_summary.index
) == set(region_order)

assert dashboard_summary[
    [
        "MAE_Ft",
        "MAPE_%",
        "Bias_Ft"
    ]
].notna().all().all()


display(
    dashboard_summary[
        [
            "Klaszter_neve",
            "MAE_Ft",
            "MAPE_%",
            "Bias_Ft"
        ]
    ]
    .sort_values("MAE_Ft")
    .round(2)
)


print(
    "A dashboard területi összefoglaló "
    "adatai elkészültek."
)

,Klaszter_neve,MAE_Ft,MAPE_%,Bias_Ft
Területi egység neve,,,,
Baranya,Alacsonyabb bérszint / magasabb munkanélküliség,5693.40,0.98,4451.65
Békés,Alacsonyabb bérszint / magasabb munkanélküliség,6557.07,1.30,6557.07
Jász-Nagykun-Szolnok,Alacsonyabb bérszint / magasabb munkanélküliség,7155.54,1.29,6373.62
Szabolcs-Szatmár-Bereg,Alacsonyabb bérszint / magasabb munkanélküliség,7250.80,1.44,4956.43
Somogy,Alacsonyabb bérszint / magasabb munkanélküliség,8311.60,1.50,7972.21
Csongrád-Csanád,Magasabb bérszint / alacsonyabb munkanélküliség,8453.01,1.44,6208.92
Bács-Kiskun,Magasabb bérszint / alacsonyabb munkanélküliség,8702.32,1.49,7227.64
Pest,Magasabb bérszint / alacsonyabb munkanélküliség,8959.67,1.46,7197.65
Hajdú-Bihar,Alacsonyabb bérszint / magasabb munkanélküliség,9030.58,1.54,5118.62


A dashboard területi összefoglaló adatai elkészültek.


In [12]:
fig_dashboard = make_subplots(
    rows=3,
    cols=3,

    specs=[
        [
            {"type": "indicator"},
            {"type": "indicator"},
            {"type": "indicator"}
        ],

        [
            {
                "type": "xy",
                "colspan": 3
            },
            None,
            None
        ],

        [
            {
                "type": "xy",
                "colspan": 3
            },
            None,
            None
        ]
    ],

    row_heights=[
        0.18,
        0.50,
        0.32
    ],

    vertical_spacing=0.10,
    shared_xaxes=True
)


dashboard_trace_indices = {}


for region_name in region_order:

    region_data = (
        ridge_time_data.loc[
            ridge_time_data[
                "Területi egység neve"
            ]
            ==
            region_name
        ]
        .sort_values("Cél_Dátum")
    )


    region_summary = (
        dashboard_summary.loc[
            region_name
        ]
    )


    is_default_region = (
        region_name == DEFAULT_REGION
    )


    first_trace_index = len(
        fig_dashboard.data
    )


    # 1. KPI – MAE

    fig_dashboard.add_trace(
        go.Indicator(
            mode="number",

            value=float(
                region_summary["MAE_Ft"]
            ),

            title={
                "text": "Átlagos abszolút hiba – MAE"
            },

            number={
                "valueformat": ",.0f",
                "suffix": " Ft"
            },

            visible=is_default_region
        ),

        row=1,
        col=1
    )


    # 2. KPI – MAPE

    fig_dashboard.add_trace(
        go.Indicator(
            mode="number",

            value=float(
                region_summary["MAPE_%"]
            ),

            title={
                "text": "Relatív hiba – MAPE"
            },

            number={
                "valueformat": ".2f",
                "suffix": "%"
            },

            visible=is_default_region
        ),

        row=1,
        col=2
    )


    # 3. KPI – BIAS

    fig_dashboard.add_trace(
        go.Indicator(
            mode="number",

            value=float(
                region_summary["Bias_Ft"]
            ),

            title={
                "text": (
                    "Irányított hiba – bias"
                    "<br><span style='font-size:12px'>"
                    "+ = felülbecslés"
                    "</span>"
                )
            },

            number={
                "valueformat": "+,.0f",
                "suffix": " Ft"
            },

            visible=is_default_region
        ),

        row=1,
        col=3
    )


    # 4. VALÓS BÉR IDŐSORA

    fig_dashboard.add_trace(
        go.Scatter(
            x=region_data["Cél_Dátum"],
            y=region_data["Valós_bér"],

            customdata=region_data[
                ["Negyedév"]
            ].to_numpy(),

            mode="lines+markers",
            name="Valós bér",

            line={
                "color": "#111111",
                "width": 3
            },

            marker={
                "size": 8,
                "symbol": "circle"
            },

            visible=is_default_region,

            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Valós bér: %{y:,.0f} Ft"
                "<extra>Valós bér</extra>"
            )
        ),

        row=2,
        col=1
    )


    # 5. RIDGE-PREDIKCIÓ IDŐSORA

    fig_dashboard.add_trace(
        go.Scatter(
            x=region_data["Cél_Dátum"],
            y=region_data["Predikció"],

            customdata=region_data[
                [
                    "Negyedév",
                    "Hiba",
                    "APE_%"
                ]
            ].to_numpy(),

            mode="lines+markers",
            name="Ridge-predikció",

            line={
                "color": "#0072B2",
                "width": 3,
                "dash": "dash"
            },

            marker={
                "size": 8,
                "symbol": "square"
            },

            visible=is_default_region,

            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Ridge-predikció: %{y:,.0f} Ft<br>"
                "Irányított hiba: "
                "%{customdata[1]:+,.0f} Ft<br>"
                "APE: %{customdata[2]:.2f}%"
                "<extra>Ridge-predikció</extra>"
            )
        ),

        row=2,
        col=1
    )


    # 6. NEGYEDÉVES ABSZOLÚT HIBA

    fig_dashboard.add_trace(
        go.Bar(
            x=region_data["Cél_Dátum"],
            y=region_data["Abszolút_hiba"],

            customdata=region_data[
                [
                    "Negyedév",
                    "Hiba",
                    "APE_%"
                ]
            ].to_numpy(),

            name="Abszolút hiba",

            marker={
                "color": "#E69F00"
            },

            visible=is_default_region,

            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Abszolút hiba: %{y:,.0f} Ft<br>"
                "Irányított hiba: "
                "%{customdata[1]:+,.0f} Ft<br>"
                "APE: %{customdata[2]:.2f}%"
                "<extra>Abszolút hiba</extra>"
            )
        ),

        row=3,
        col=1
    )


    dashboard_trace_indices[
        region_name
    ] = list(
        range(
            first_trace_index,
            len(fig_dashboard.data)
        )
    )

In [13]:
dashboard_buttons = []


for region_name in region_order:

    visibility_mask = [
        False
    ] * len(fig_dashboard.data)


    for trace_index in (
        dashboard_trace_indices[
            region_name
        ]
    ):
        visibility_mask[
            trace_index
        ] = True


    dashboard_buttons.append(
        {
            "label": region_name,
            "method": "update",

            "args": [
                {
                    "visible":
                        visibility_mask
                },

                {
                    "title.text": (
                        "A Ridge-modell interaktív "
                        "eredménydashboardja – "
                        + region_name
                    ),

                    "yaxis.autorange": True,
                    "yaxis2.autorange": True
                }
            ]
        }
    )


fig_dashboard.update_layout(
    template="plotly_white",

    title={
        "text": (
            "A Ridge-modell interaktív "
            "eredménydashboardja – "
            + DEFAULT_REGION
        ),
        "x": 0.5,
        "xanchor": "center"
    },

    updatemenus=[
        {
            "buttons": dashboard_buttons,
            "active": 0,
            "direction": "down",
            "showactive": True,
            "x": 0.14,
            "y": 1.09,
            "xanchor": "left",
            "yanchor": "top"
        }
    ],

    annotations=[
        {
            "text": "<b>Területi egység:</b>",
            "x": 0,
            "y": 1.065,
            "xref": "paper",
            "yref": "paper",
            "showarrow": False,
            "xanchor": "left"
        }
    ],

    legend={
        "orientation": "h",
        "x": 1,
        "y": 0.76,
        "xanchor": "right",
        "yanchor": "bottom"
    },

    hovermode="x unified",

    barmode="group",

    margin={
        "l": 90,
        "r": 45,
        "t": 150,
        "b": 90
    },

    height=980
)


# IDŐSOROS PANEL TENGELYEI

fig_dashboard.update_yaxes(
    title_text="Bruttó átlagbér (Ft)",
    tickformat=",.0f",
    autorange=True,
    row=2,
    col=1
)


fig_dashboard.update_xaxes(
    tickmode="array",
    tickvals=target_dates["Cél_Dátum"],
    ticktext=target_dates["Negyedév"],
    showticklabels=False,
    row=2,
    col=1
)


# HIBAPANEL TENGELYEI

fig_dashboard.update_yaxes(
    title_text="Abszolút hiba (Ft)",
    tickformat=",.0f",
    rangemode="tozero",
    row=3,
    col=1
)


fig_dashboard.update_xaxes(
    title_text="Előrejelzett célnegyedév",
    tickmode="array",
    tickvals=target_dates["Cél_Dátum"],
    ticktext=target_dates["Negyedév"],
    tickangle=-35,
    row=3,
    col=1
)


# SZERKEZETI ELLENŐRZÉSEK

assert len(fig_dashboard.data) == 120
assert len(dashboard_buttons) == 20

assert all(
    len(
        dashboard_trace_indices[
            region_name
        ]
    )
    ==
    6

    for region_name in region_order
)

assert all(
    sum(
        button["args"][0]["visible"]
    )
    ==
    6

    for button in dashboard_buttons
)


fig_dashboard.show()

In [14]:
DASHBOARD_HTML = (
    INTERACTIVE_FIGURES
    / "03_interaktiv_ridge_eredmenydashboard.html"
)


fig_dashboard.write_html(
    DASHBOARD_HTML,
    include_plotlyjs=True,
    full_html=True
)


dashboard_summary.reset_index().to_csv(
    INTERACTIVE_RESULTS
    / "KSH_interaktiv_dashboard_teruleti_osszefoglalo.csv",
    index=False,
    encoding="utf-8-sig"
)


assert DASHBOARD_HTML.exists()


print(
    "Az interaktív eredménydashboard elmentve:",
    DASHBOARD_HTML
)

Az interaktív eredménydashboard elmentve: c:\Users\wldbe\Downloads\szakdolgozat12-20260820T154901Z-1-001\szakdolgozat12\figures\interactive\03_interaktiv_ridge_eredmenydashboard.html


## 5.4. Az interaktív dashboard eredményeinek értelmezése

Az interaktív dashboard a végleges Ridge-modell területi
teljesítményének részletes vizsgálatát támogatja. A három
összefoglaló mutató eltérő szempontból jellemzi az előrejelzési
hibát.

A MAE az előrejelzés és a tényleges bér közötti eltérés átlagos
forintértékét mutatja. A MAPE ugyanezt a terület bérszintjéhez
viszonyítva, százalékos formában fejezi ki. A bias az előrejelzési
hiba irányát mutatja: pozitív értéke átlagos felülbecslést, negatív
értéke pedig átlagos alulbecslést jelent.

A mutatókat ezért együttesen kell értelmezni. Egy magasabb bérszintű
terület abszolút hibája nagyobb lehet úgy is, hogy a relatív
százalékos hiba továbbra is alacsony marad. Hasonlóképpen az
alacsony bias nem feltétlenül jelent pontos előrejelzést, mert az
ellentétes irányú hibák részben kiolthatják egymást.

In [15]:
best_mae_region = (
    dashboard_summary[
        "MAE_Ft"
    ]
    .idxmin()
)


worst_mae_region = (
    dashboard_summary[
        "MAE_Ft"
    ]
    .idxmax()
)


highest_mape_region = (
    dashboard_summary[
        "MAPE_%"
    ]
    .idxmax()
)


highest_bias_region = (
    dashboard_summary[
        "Bias_Ft"
    ]
    .idxmax()
)


positive_bias_regions = int(
    dashboard_summary[
        "Bias_Ft"
    ]
    .gt(0)
    .sum()
)


notebook_5_summary = pd.DataFrame(
    [
        {
            "Mutató":
                "Interaktívan vizsgált területek",

            "Eredmény":
                dashboard_summary.shape[0]
        },

        {
            "Mutató":
                "Vizsgált célnegyedévek",

            "Eredmény":
                ridge_time_data[
                    "Cél_Dátum"
                ].nunique()
        },

        {
            "Mutató":
                "Legkisebb területi MAE",

            "Eredmény": (
                f"{best_mae_region} – "
                +
                f"{dashboard_summary.loc[best_mae_region, 'MAE_Ft']:,.0f} Ft"
                .replace(",", " ")
            )
        },

        {
            "Mutató":
                "Legnagyobb területi MAE",

            "Eredmény": (
                f"{worst_mae_region} – "
                +
                f"{dashboard_summary.loc[worst_mae_region, 'MAE_Ft']:,.0f} Ft"
                .replace(",", " ")
            )
        },

        {
            "Mutató":
                "Legnagyobb területi MAPE",

            "Eredmény": (
                f"{highest_mape_region} – "
                +
                f"{dashboard_summary.loc[highest_mape_region, 'MAPE_%']:.2f}%"
            )
        },

        {
            "Mutató":
                "Legnagyobb átlagos felülbecslés",

            "Eredmény": (
                f"{highest_bias_region} – "
                +
                f"{dashboard_summary.loc[highest_bias_region, 'Bias_Ft']:,.0f} Ft"
                .replace(",", " ")
            )
        },

        {
            "Mutató":
                "Pozitív átlagos biasszal rendelkező területek",

            "Eredmény": (
                f"{positive_bias_regions} / "
                f"{len(dashboard_summary)}"
            )
        }
    ]
)


display(notebook_5_summary)

,Mutató,Eredmény
0,Interaktívan vizsgált területek,20
1,Vizsgált célnegyedévek,8
2,Legkisebb területi MAE,Baranya – 5 693 Ft
3,Legnagyobb területi MAE,Tolna – 18 495 Ft
4,Legnagyobb területi MAPE,Tolna – 2.92%
5,Legnagyobb átlagos felülbecslés,Győr-Moson-Sopron – 17 816 Ft
6,Pozitív átlagos biasszal rendelkező területek,20 / 20


### Területi következtetések

A végleges tesztidőszakban Baranya rendelkezett a legkisebb
területi előrejelzési hibával: a MAE értéke megközelítőleg
5,7 ezer forint, a MAPE értéke pedig 0,98% volt. Ezzel szemben
Tolna esetében körülbelül 18,5 ezer forintos MAE és 2,92%-os
MAPE jelentkezett. Győr-Moson-Sopron szintén a nehezebben
előrejelezhető területek közé tartozott, 18,4 ezer forint körüli
MAE-értékkel.

Budapest 11,6 ezer forint körüli abszolút hibája magasabb volt
az országos területi értékek jelentős részénél, ugyanakkor a
MAPE mindössze 1,44%-ot tett ki. Ez azt mutatja, hogy a magasabb
bérszint részben növeli a forintban kifejezett hibát, miközben
a relatív előrejelzési pontosság továbbra is kedvező lehet.

Tolna példája a MAE és a bias eltérő jelentését is szemlélteti.
A terület MAE-értéke volt a legmagasabb, miközben az átlagos
bias körülbelül 4,1 ezer forintot tett ki. Ez arra utal, hogy a
nagyobb pozitív és negatív hibák részben kioltották egymást az
irányított átlag számításakor. Emiatt a bias önmagában nem
használható az előrejelzési pontosság mérésére.

A területi átlagos bias mind a húsz vizsgált egység esetében
pozitív volt, vagyis a Ridge-modell a 2024–2025-ös tesztidőszak
egészét tekintve általánosan felülbecslő tendenciát mutatott.
Az idősoros panel ugyanakkor megmutatja, hogy a hiba nagysága és
iránya negyedévenként változott, ezért az átlagos területi
mutatókat az időbeli lefutással együtt kell értelmezni.

Az eredmények leíró és előrejelzési összefüggéseket mutatnak.
A dashboard alapján önmagában nem állapítható meg, hogy valamely
gazdasági tényező okozta az előrejelzési hibák területi
különbségeit.

## 5.5. Interaktív kimenetek és záró ellenőrzés

A notebook három önálló interaktív vizualizációt készített:

1. a területi előrejelzési hibák NUTS 3 térképét;
2. a területenként választható valós–predikció idősorát;
3. a területi KPI-kat, idősorokat és negyedéves hibákat
   egyesítő eredménydashboardot.

A kimenetek önálló HTML-fájlként is megnyithatók, így a
szakdolgozat elektronikus mellékleteként és a védés során is
felhasználhatók.

In [16]:
MAP_DATA_FILE = (
    INTERACTIVE_RESULTS
    / "KSH_interaktiv_terkepi_adatok.csv"
)


NOTEBOOK_5_SUMMARY_FILE = (
    INTERACTIVE_RESULTS
    / "KSH_notebook_5_eredmenyek_osszefoglalasa.csv"
)


regional_map_data.to_csv(
    MAP_DATA_FILE,
    index=False,
    encoding="utf-8-sig"
)


notebook_5_summary.to_csv(
    NOTEBOOK_5_SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig"
)


print(
    "A térképadatok és az eredményösszefoglaló elmentve."
)

A térképadatok és az eredményösszefoglaló elmentve.


In [17]:
interactive_output_manifest = pd.DataFrame(
    [
        {
            "Sorszám": 1,
            "Kimenet":
                "Interaktív területi hibatérkép",
            "Formátum":
                "HTML",
            "Fájl":
                str(MAP_HTML)
        },

        {
            "Sorszám": 2,
            "Kimenet":
                "Területválasztós valós–predikció idősor",
            "Formátum":
                "HTML",
            "Fájl":
                str(REGION_CHART_HTML)
        },

        {
            "Sorszám": 3,
            "Kimenet":
                "Interaktív Ridge-eredménydashboard",
            "Formátum":
                "HTML",
            "Fájl":
                str(DASHBOARD_HTML)
        },

        {
            "Sorszám": 4,
            "Kimenet":
                "Interaktív térkép területi adatai",
            "Formátum":
                "CSV",
            "Fájl":
                str(MAP_DATA_FILE)
        },

        {
            "Sorszám": 5,
            "Kimenet":
                "Területenkénti idősoros adatok",
            "Formátum":
                "CSV",
            "Fájl":
                str(
                    INTERACTIVE_RESULTS
                    / "KSH_interaktiv_teruleti_idosor_adatok.csv"
                )
        },

        {
            "Sorszám": 6,
            "Kimenet":
                "Dashboard területi összefoglaló",
            "Formátum":
                "CSV",
            "Fájl":
                str(
                    INTERACTIVE_RESULTS
                    / "KSH_interaktiv_dashboard_teruleti_osszefoglalo.csv"
                )
        },

        {
            "Sorszám": 7,
            "Kimenet":
                "Eurostat GISCO NUTS 3 geometria",
            "Formátum":
                "GeoJSON",
            "Fájl":
                str(GEOJSON_FILE)
        }
    ]
)


interactive_output_manifest[
    "Létezik"
] = interactive_output_manifest[
    "Fájl"
].map(
    lambda file_path:
        Path(file_path).exists()
)


interactive_output_manifest[
    "Fájlméret_MB"
] = interactive_output_manifest[
    "Fájl"
].map(
    lambda file_path:
        Path(file_path).stat().st_size
        /
        1024 ** 2
)


assert len(interactive_output_manifest) == 7

assert interactive_output_manifest[
    "Létezik"
].all()


display(
    interactive_output_manifest[
        [
            "Sorszám",
            "Kimenet",
            "Formátum",
            "Létezik",
            "Fájlméret_MB"
        ]
    ]
    .round(
        {
            "Fájlméret_MB": 3
        }
    )
)


interactive_output_manifest.to_csv(
    INTERACTIVE_RESULTS
    / "KSH_notebook_5_interaktiv_fajljegyzek.csv",
    index=False,
    encoding="utf-8-sig"
)


print(
    "Az interaktív fájljegyzék elkészült."
)

,Sorszám,Kimenet,Formátum,Létezik,Fájlméret_MB
0,1,Interaktív területi hibatérkép,HTML,True,4.721
1,2,Területválasztós valós–predikció idősor,HTML,True,4.683
2,3,Interaktív Ridge-eredménydashboard,HTML,True,4.728
3,4,Interaktív térkép területi adatai,CSV,True,0.004
4,5,Területenkénti idősoros adatok,CSV,True,0.017
5,6,Dashboard területi összefoglaló,CSV,True,0.003
6,7,Eurostat GISCO NUTS 3 geometria,GeoJSON,True,6.338


Az interaktív fájljegyzék elkészült.


In [18]:
import numpy as np


# TERÜLETI MUTATÓK ÚJRASZÁMÍTÁSA

recalculated_regional_metrics = (
    ridge_time_data
    .groupby(
        "Területi egység neve"
    )
    .agg(
        Új_MAE_Ft=(
            "Abszolút_hiba",
            "mean"
        ),

        Új_MAPE_pct=(
            "APE_%",
            "mean"
        ),

        Új_Bias_Ft=(
            "Hiba",
            "mean"
        )
    )
)


metric_consistency_check = (
    dashboard_summary[
        [
            "MAE_Ft",
            "MAPE_%",
            "Bias_Ft"
        ]
    ]
    .join(
        recalculated_regional_metrics,
        how="left"
    )
)


assert np.allclose(
    metric_consistency_check[
        "MAE_Ft"
    ],
    metric_consistency_check[
        "Új_MAE_Ft"
    ]
)


assert np.allclose(
    metric_consistency_check[
        "MAPE_%"
    ],
    metric_consistency_check[
        "Új_MAPE_pct"
    ]
)


assert np.allclose(
    metric_consistency_check[
        "Bias_Ft"
    ],
    metric_consistency_check[
        "Új_Bias_Ft"
    ]
)


# ADATLEFEDETTSÉG

assert len(regional_map_data) == 20
assert regional_map_data["NUTS_ID"].nunique() == 20

assert len(ridge_time_data) == 160

assert ridge_time_data[
    "Cél_Dátum"
].nunique() == 8

assert (
    ridge_time_data
    .groupby("Területi egység neve")
    .size()
    .eq(8)
    .all()
)


# INTERAKTÍV ÁBRÁK SZERKEZETE

assert len(fig_map.data) == 1

assert len(
    fig_map.data[0].locations
) == 20

assert len(fig_region.data) == 40

assert len(
    fig_region.layout
    .updatemenus[0]
    .buttons
) == 20

assert len(fig_dashboard.data) == 120

assert len(
    fig_dashboard.layout
    .updatemenus[0]
    .buttons
) == 20


# KIMENETI FÁJLOK

assert interactive_output_manifest[
    "Létezik"
].all()


notebook_5_audit = pd.DataFrame(
    [
        {
            "Ellenőrzés":
                "A 20 területi egység térképi illesztése teljes",
            "Eredmény":
                "Megfelelt"
        },

        {
            "Ellenőrzés":
                "Mind a 160 végső Ridge-előrejelzés elérhető",
            "Eredmény":
                "Megfelelt"
        },

        {
            "Ellenőrzés":
                "Minden területhez nyolc célnegyedév tartozik",
            "Eredmény":
                "Megfelelt"
        },

        {
            "Ellenőrzés":
                "A térkép mind a 20 területet tartalmazza",
            "Eredmény":
                "Megfelelt"
        },

        {
            "Ellenőrzés":
                "A területválasztós idősor mind a 20 területet kezeli",
            "Eredmény":
                "Megfelelt"
        },

        {
            "Ellenőrzés":
                "A dashboard mind a 20 területet és 3 KPI-t kezeli",
            "Eredmény":
                "Megfelelt"
        },

        {
            "Ellenőrzés":
                "Az újraszámított területi mutatók megegyeznek",
            "Eredmény":
                "Megfelelt"
        },

        {
            "Ellenőrzés":
                "Minden interaktív kimeneti fájl elérhető",
            "Eredmény":
                "Megfelelt"
        }
    ]
)


display(notebook_5_audit)


notebook_5_audit.to_csv(
    INTERACTIVE_RESULTS
    / "KSH_notebook_5_zaro_audit.csv",
    index=False,
    encoding="utf-8-sig"
)


print("=" * 80)

print(
    "Az 5. notebook minden ellenőrzést "
    "teljesített és lezárható."
)

print("=" * 80)

,Ellenőrzés,Eredmény
0,A 20 területi egység térképi illesztése teljes,Megfelelt
1,Mind a 160 végső Ridge-előrejelzés elérhető,Megfelelt
2,Minden területhez nyolc célnegyedév tartozik,Megfelelt
3,A térkép mind a 20 területet tartalmazza,Megfelelt
4,A területválasztós idősor mind a 20 területet ...,Megfelelt
5,A dashboard mind a 20 területet és 3 KPI-t kezeli,Megfelelt
6,Az újraszámított területi mutatók megegyeznek,Megfelelt
7,Minden interaktív kimeneti fájl elérhető,Megfelelt


Az 5. notebook minden ellenőrzést teljesített és lezárható.


## 5.6. Összegzés

Az interaktív vizualizációk a statikus eredményábrákat
területi és időbeli szempontból egészítik ki. A NUTS 3
hibatérkép a modell teljesítményének földrajzi különbségeit,
a területválasztós idősor a valós és előrejelzett bérek
negyedéves eltéréseit, az eredménydashboard pedig a legfontosabb
hibamutatókat és azok időbeli szerkezetét mutatja be.

A vizualizációk alapján a Ridge-modell összességében magas
pontosságot ért el, ugyanakkor a hibák nagysága területenként
és negyedévenként eltért. Az eredmények ezért nemcsak országos
átlagban, hanem területi és időbeli bontásban is értelmezendők.

A notebook interaktív kimenetei a szakdolgozat elektronikus
mellékleteként és a védés vizuális támogatására is
felhasználhatók.